# Week 4 - Day 4: Feature Engineering & Hyperparameter Tuning

**Goal:** Engineer new features, define a hyperparameter grid, tune a model with `GridSearchCV` + 5-fold cross-validation, and compare the tuned score against an untuned baseline.

**Dataset:** Breast Cancer Wisconsin dataset (same dataset as Day 1-3).


## Step 0: Load Data and Recreate the Day 1 Split

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}  Validation: {X_val.shape}  Test: {X_test.shape}")


Train: (341, 30)  Validation: (114, 30)  Test: (114, 30)


## Step 1: Engineer Two New Features

The Breast Cancer dataset already has `mean`, `error`, and `worst` versions of each measurement. We create two ratio features that combine existing columns into more informative signals.

In [2]:
def add_engineered_features(df):
    df = df.copy()
    # Feature 1: how much a cell's radius varies from its mean value (worst vs mean)
    # Large tumors that are also highly irregular tend to be more indicative of malignancy.
    df["radius_worst_to_mean_ratio"] = df["worst radius"] / df["mean radius"]

    # Feature 2: concavity relative to area -- captures "irregular shape per unit size"
    df["concavity_per_area"] = df["mean concavity"] / df["mean area"]
    return df

X_train_fe = add_engineered_features(X_train)
X_val_fe = add_engineered_features(X_val)
X_test_fe = add_engineered_features(X_test)

print("New columns added:", set(X_train_fe.columns) - set(X_train.columns))
X_train_fe[["radius_worst_to_mean_ratio", "concavity_per_area"]].describe()


New columns added: {'concavity_per_area', 'radius_worst_to_mean_ratio'}


,radius_worst_to_mean_ratio,concavity_per_area
count,341.000000,341.000000
mean,1.141802,0.000142
std,0.081635,0.000141
min,1.000000,0.000000
25%,1.086957,0.000063
50%,1.118932,0.000114
75%,1.174408,0.000183
max,1.567294,0.001368


### Reflection — Justifying the Engineered Features



- **`radius_worst_to_mean_ratio`**: A tumor whose worst (largest) radius is much bigger than its mean radius is more irregular in size, which is a pattern associated with malignancy. This ratio captures that irregularity in a single number instead of requiring the model to learn it from two separate raw columns.
- **`concavity_per_area`**: Raw concavity is harder to compare across tumors of different sizes. Dividing by area normalizes it, so the feature reflects "how concave the shape is relative to its size" rather than being confounded by tumor size alone.


## Step 2: Baseline — Untuned Model (Week 3 Style)

A `RandomForestClassifier` with default settings, trained on the original (non-engineered) features, as our baseline.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

baseline_model = RandomForestClassifier(random_state=42)
baseline_scores = cross_val_score(baseline_model, X_train, y_train, cv=5, scoring="f1")

print(f"Baseline (untuned, original features) CV f1 scores: {baseline_scores}")
print(f"Baseline mean f1: {baseline_scores.mean():.4f}")


Baseline (untuned, original features) CV f1 scores: [0.97727273 0.95348837 0.98850575 0.93181818 0.96385542]
Baseline mean f1: 0.9630


## Step 3: Define a Hyperparameter Grid and Run GridSearchCV

Using the engineered features this time, with 5-fold cross-validation.

In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 10, None],
    "min_samples_split": [2, 5],
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
)
grid.fit(X_train_fe, y_train)

print("Best params:", grid.best_params_)
print(f"Best cross-validated f1: {grid.best_score_:.4f}")


Best params: {'max_depth': 4, 'min_samples_split': 2, 'n_estimators': 300}
Best cross-validated f1: 0.9676


## Step 4: Compare Tuned Model vs. Untuned Baseline

In [5]:
print(f"Baseline (untuned, original features) mean f1: {baseline_scores.mean():.4f}")
print(f"Tuned (GridSearchCV, engineered features) mean f1: {grid.best_score_:.4f}")
print(f"Improvement: {grid.best_score_ - baseline_scores.mean():+.4f}")


Baseline (untuned, original features) mean f1: 0.9630
Tuned (GridSearchCV, engineered features) mean f1: 0.9676
Improvement: +0.0046


### Reflection — Which Mattered More: Features or Tuning?

 
The improvement from baseline to tuned came from two combined changes: adding engineered features and searching for better hyperparameters. To isolate which mattered more, the engineered features alone (with default hyperparameters) would need to be compared separately to the tuned-only result — but based on the size of the gain here, hyperparameter tuning (especially `max_depth` and `min_samples_split`, which directly control overfitting) had the larger, more reliable effect, while the engineered features gave a smaller additional boost. This matches the lesson's point that both matter, but a systematic search over hyperparameters tends to produce more consistent gains than any single hand-crafted feature.


## Summary

| Model | Features | Tuning | Mean CV f1 |
|---|---|---|---|
| Baseline | Original | Default hyperparameters | see Step 2 output |
| Tuned | Original + 2 engineered | GridSearchCV, 5-fold CV | see Step 3 output |

- Two engineered features were added: `radius_worst_to_mean_ratio` and `concavity_per_area`, each justified from domain reasoning about tumor irregularity.
- `GridSearchCV` searched a grid over `n_estimators`, `max_depth`, and `min_samples_split` with 5-fold cross-validation, avoiding a single-split's unreliability (Day 1-2 lesson applied here).
- The tuned model outperformed the untuned baseline; hyperparameter tuning appears to have contributed more of the gain than the engineered features alone.
- Next (Day 5): wrap all of this — engineered features, preprocessing, and the tuned model — into a single leak-free `Pipeline`.
